In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

In [2]:
df = pd.read_csv(
    '../../../data/nyc-parking-violations-2020.csv',
    engine = 'pyarrow', # here, using pyarrow makes reading the DataFrame about ~5x faster, this was suggested by Claude
    dtype_backend = 'pyarrow', 
    usecols = ['Plate ID', 'Registration State', 'Vehicle Make', 'Vehicle Color', 'Street Name']
)

In [3]:
df

,Plate ID,Registration State,Vehicle Make,Vehicle Color,Street Name
0,J58JKX,NJ,HONDA,BK,43 ST
1,KRE6058,PA,ME/BE,BLK,UNION ST
2,444326R,NJ,LEXUS,BLACK,CLERMONT AVENUE
3,F728330,OH,CHEVR,<NA>,DIVISION AVE
4,FMY9090,NY,JEEP,GREY,GRAND ST
...,...,...,...,...,...
12495729,62161MM,NY,FORD,BR,3RD AVE
12495730,GYE7330,NY,HONDA,BLK,PELHAM PARK DR
12495731,HNY4802,NY,FORD,GY,LYDIG AVE
12495732,T687081C,NY,TOYOT,BLK,E 68 STREET


In [4]:
len(df['Vehicle Color'].value_counts())

1896

In [5]:
df['Vehicle Color'].value_counts().head(30).index.to_list()

['WH',
 'GY',
 'BK',
 'WHITE',
 'BL',
 'RD',
 'BLACK',
 'GREY',
 'BROWN',
 'SILVE',
 'GR',
 'BLUE',
 'RED',
 'TN',
 'BR',
 'YW',
 'BLK',
 'OTHER',
 'GREEN',
 'GL',
 'GRY',
 'MR',
 'GRAY',
 'WHT',
 'YELLO',
 'WHI',
 'OR',
 'BK.',
 'WT',
 'WT.']

In [6]:
color_map = {
    # WHITE
    'WH':    'WHITE',
    'WHT':   'WHITE',
    'WHI':   'WHITE',
    'WHITE': 'WHITE',
    'WT':    'WHITE',
    'WT.':   'WHITE',

    # BLACK
    'BK':    'BLACK',
    'BK.':   'BLACK',
    'BLK':   'BLACK',
    'BLACK': 'BLACK',

    # GRAY / GREY
    'GY':   'GRAY',
    'GR':   'GRAY',   # ⚠️ ambiguous — could be GREEN
    'GRY':  'GRAY',
    'GREY': 'GRAY',
    'GRAY': 'GRAY',

    # BLUE
    'BL':   'BLUE',
    'BLUE': 'BLUE',

    # RED
    'RD':  'RED',
    'RED': 'RED',

    # BROWN
    'BR':    'BROWN',
    'BROWN': 'BROWN',

    # GREEN
    'GREEN': 'GREEN',

    # SILVER
    'SILVE': 'SILVER',

    # TAN
    'TN': 'TAN',

    # YELLOW
    'YW':    'YELLOW',
    'YELLO': 'YELLOW',

    # ORANGE
    'OR': 'ORANGE',

    # GOLD  ⚠️ uncertain — GL could also mean GLASS or SILVER
    'GL': 'GOLD',

    # MAROON  ⚠️ uncertain — MR could also mean MIRROR or something else
    'MR': 'MAROON',

    # OTHER
    'OTHER': 'OTHER',
}

In [7]:
df_2 = df.copy()

In [8]:
df_2['Vehicle Color'] = df_2['Vehicle Color'].replace(color_map)

In [9]:
df_2['Vehicle Color'].value_counts().reset_index()

# The number of different color categories decreased by only 18,
# and many colors remain unidentified or difficult to identify.

,Vehicle Color,count
0,WHITE,3521461
1,GRAY,2884801
2,BLACK,2650853
3,BLUE,953422
4,RED,644991
...,...,...
1873,GLKPP,1
1874,';,1
1875,WHGY8,1
1876,ORYW,1


#### Beyon the exercise_1
##### Run value_counts on the Vehicle Make column, and look at some vehicle names. (There are more than 5,200 distinct makes, which almost certainly indicates a lot of inconsistency in this data.)
##### What problems do you see?
##### Write a function that, given a value, cleans up the data: putting the name in all caps, removing punctuation, and standardizing whatever names you can. Then use the apply method to fix the column. How many distinct vehicle makes are there when you’re done?

In [10]:
pd.set_option('display.max_rows', 20)

df['Vehicle Make'].value_counts().reset_index().head(20)

# up to 5 characters can be entered for the Vehicle Make.

,Vehicle Make,count
0,TOYOT,1395273
1,HONDA,1343265
2,FORD,1328063
3,NISSA,1119587
4,CHEVR,711464
5,FRUEH,530846
6,ME/BE,530473
7,JEEP,490977
8,BMW,488545
9,DODGE,462646


In [11]:
import string


def clean_vehicle_make(vm):
    if pd.isna(vm):
        vm = 'UNKNOWN'

    vm = vm.upper()

    ascii_punctuation = list(string.punctuation)
    for i in ascii_punctuation:
        vm = vm.replace(i, '')
    
    return vm

In [12]:
df['Vehicle Make'] = df['Vehicle Make'].apply(clean_vehicle_make)

In [13]:
df['Vehicle Make'] = df.groupby(
    df['Vehicle Make'].str[:3]
)['Vehicle Make'].transform(
    lambda x: x.value_counts().idxmax()
)

In [14]:
df['Vehicle Make'].nunique()

2370

In [15]:
df['Vehicle Make'].value_counts()

Vehicle Make
TOYOT    1396758
HONDA    1344485
FORD     1329522
NISSA    1121810
CHEVR     712986
          ...   
LNPR           1
GOTLE          1
DOFA           1
KHA            1
II8            1
Name: count, Length: 2370, dtype: int64